# New-RscMssqlSnapshot
DBAs will often take a backup before they make a change to the database. In general, you need 2 things to take a snapshot of a database, the database ID and the SLA ID. An SLA is required for all on demand snapshots to manage the retention of the snapshot. 

# Workflow
To start an On-Demand Snapsot of a database, you need to do the following
1. Use **Connect-RSC** to Connect to RSC
1. Use **Get-RscCluster** to get the cluster id that the SQL Server is registered. This is akin to the datacenter that the SQL Server is located in. While you can run the next command without the cluster ID, you may get more records back than expected. This is because your SQL Server Host could exist on 2 different cluster. Once as a replicated source and once as the replicated target. 
1. Use **Get-RscMssqlInstance** to get the instance id of the SQL Server based on the name of the instance and the cluster where it is located. 
1. Use **Get-RscMssqlDatabase** to get the database ID based on the database name and the SQL Server Instance. 
1. Use **Get-RscSLADomain** to get the ID SLA Domain that will manage the retention of the snapshot.
1. Use **New-RscMssqlSnapshot** to initiate an On Demand Snapshot of a database. 
1. Use **Get-RscRequest** to track the request and know when it completes. ***DOES NOT EXIST YET***
1. Use **Disconnect-RSC** to disconnect from RSC

# Connect to RSC

In [6]:
Import-Module RubrikSecurityCloud
$ConnectRsc = @{
    Server = op read "op://Rubrik/RoadRunnerServiceAccount/hostname"
    ClientId = op read "op://Rubrik/RoadRunnerServiceAccount/username"
    ClientSecret = ConvertTo-SecureString (op read "op://Rubrik/RoadRunnerServiceAccount/credential") -AsPlainText -Force
}
Connect-RSC @ConnectRsc

ServiceAccountDetails


ClientId,client|1461d755-e4d2-4a71-8a5d-204f7d6bc73e
AuthenticationState,AUTHORIZED
SdkSchemaVersion,v20250616-31
ServerVersion,v20250623-27


# Get the Cluster ID

In [7]:
$ClusterName = "vault-r-london"
$RscCluster = Get-RscCluster -Name $ClusterName

# Get the Database based on the Instance

## Get the Instance Information

In [12]:
$HostName = "rp-sql22ags-1a.perf.rubrik.com"
$RscMssqlInstance = Get-RscMssqlInstance -RscCluster $RscCluster -HostName $HostName

## Get the Database Information

In [13]:
$DatabaseName = "master"
$RscMssqlDatabase = Get-RscMssqlDatabase -Name $DatabaseName -RscMssqlInstance $RscMssqlInstance -Replica:$false -Relic:$false
$RscMssqlDatabase

RubrikSecurityCloud.Types.MssqlDatabase AuthorizedOperations <null> CdmPendingObjectPauseAssignment <null> ObjectType MSSQL SlaAssignment DERIVED ConfiguredSlaDomain <null> EffectiveRetentionSlaDomain <null> EffectiveSlaDomain RubrikSecurityCloud.Types.GlobalSlaReply ObjectTypes <null> RetentionLockMode <null> ClusterUuid 00000000-0000-0000-0000-000000000000 Description Multi-DC AG Support. Replication between London and Houghton Id 484fd128-d88d-4e67-955f-e6ba14e6d230 IsArchived False IsDefault False IsReadOnly False IsRetentionLockedSla False Name MSSQL-1d-30d-NoArchive-Repl-Houghton-London OwnerOrgName <null> ProtectedObjectCount <null> SnapshotScheduleLastUpdatedAt <null> StateVersion 2 UiColor <null> Version AllOrgsHavingAccess <null> AllOrgsWithAccess <null> ArchivalLocationsUpgradeInfo <null> ArchivalSpec <null> ArchivalSpecs <null> BackupWindows <null> BaseFrequency <null> ClusterToSyncStatusMap <null> FirstFullBackupWindows <null> LocalRetentionLimit <null> LogConfig <null> ObjectSpecificConfigs <null> OwnerOrg <null> PausedClustersInfo <null> ReplicationSpec <null> ReplicationSpecsV2 <null> SnapshotSchedule <null> SourceClusters <null> UpgradeInfo <null> PendingSla <null> ReplicatedObjects <null> CdmId f65702d7-5a34-48fb-b8b0-c0f4e44cc0df CdmLink <null> CdmOnDemandSnapshotCount <null> CopyOnly False DagId d389e0e9-b7e7-5fe2-8370-8f85d55ca97b HasLogConfigFromSla True HasPermissions True HostLogRetention -1 Id 65bdeaec-c5e9-5c26-b03c-6954135a3f93 IsInAvailabilityGroup False IsLogShippingSecondary False IsMount False IsOnline True IsRelic False LogBackupFrequencyInSeconds -2 LogBackupRetentionInHours -3 Name master NumWorkloadDescendants 0 OnDemandSnapshotCount <null> PostBackupScript PreBackupScript RecoveryModel <null> ReplicatedObjectCount <null> SlaPauseStatus False UnprotectableReasons <null> Version 16.0.1000.6 AllOrgs <null> AllTags <null> CdmGroupedSnapshots <null> CdmNewestSnapshot <null> CdmOldestSnapshot <null> CdmSnapshots <null> Cluster RubrikSecurityCloud.Types.Cluster CyberEventLockdownMode <null> EosStatus <null> LicensedProducts <null> PauseStatus <null> ProductType <null> RegisteredMode <null> Status <null> StatusFromDb <null> SubStatus <null> SystemStatus <null> Type <null> CdmRbacMigrationStatus <null> ConnectivityLastUpdated <null> DefaultAddress <null> DefaultPort <null> EncryptionEnabled <null> EosDate <null> EstimatedRunway <null> Id 39b92c18-d897-4b55-a7f9-17ff178616d0 IsAirGapped <null> IsClusterRemovalTprEnabled <null> IsHealthy <null> IsTprEnabled <null> LastConnectionTime <null> Name vault-r-london NoSqlWorkloadCount <null> PassesConnectivityCheck <null> RawAddress <null> RegistrationTime <null> SnapshotCount <null> SystemStatusMessage <null> Timezone <null> Version <null> ActivitySeriesConnection <null> AllOrgs <null> AuthorizedOperations <null> CcprovisionInfo <null> CdmClusterNodeDetails <null> CdmNotificationSettings <null> CdmUpgradeInfo <null> CloudInfo <null> ClusterDiskConnection <null> ClusterNodeConnection <null> ClusterNodeStats <null> ConfigProtectionInfo <null> CyberEventLockdownSupportCaseDetails <null> DatagovAutoEnablePolicyConfig <null> DatagovPreviewerConfig <null> GeoLocation <null> GlobalManagerConnectivityStatus <null> IpmiInfo <null> LambdaConfig <null> LambdaFeatureHistory <null> MetadataPullScheduler <null> Metric <null> MetricTimeSeries <null> MetricTimeSeriesNew <null> ReplicationSources <null> ReplicationTargets <null> RubrikSyncStatus <null> SnappableConnection <null> State <null> SystemStatusAffectedNodes <null> WebServerCertificate <null> Vars RubrikSecurityCloud.Types.Cluster+InlineVars ActivitySeriesConnection key value first <null> after <null> last <null> before <null> sortBy <null> filters <null> ClusterDiskConnection key value first <null> after <null> last <null> before <null> filter <null> ClusterNodeConnection key value first <null> after <null> last <null> before <null> filter <null> sortBy <null> sortOrder <null> ClusterNodeStats key value cdmClusterN

# Get the Database based on the AG

## Get the Availability Group 

In [8]:
$AvailabilityGroupName = "RR-SQL22-AG1"
$RscMssqlAvailabilityGroup = Get-RscMssqlAvailabilityGroup -AvailabilityGroupName $AvailabilityGroupName -RscCluster $RscCluster -Replica:$false -Relic:$false

### Get the Database Information

In [9]:
$DatabaseName = "AdventureWorksLT2019"
$RscMssqlDatabase = Get-RscMssqlDatabase -Name $DatabaseName -RscMssqlAvailabilityGroup $RscMssqlAvailabilityGroup
$RscMssqlDatabase

RubrikSecurityCloud.Types.MssqlDatabase AuthorizedOperations <null> CdmPendingObjectPauseAssignment <null> ObjectType MSSQL SlaAssignment DERIVED ConfiguredSlaDomain <null> EffectiveRetentionSlaDomain <null> EffectiveSlaDomain RubrikSecurityCloud.Types.GlobalSlaReply ObjectTypes <null> RetentionLockMode <null> ClusterUuid 00000000-0000-0000-0000-000000000000 Description Multi-DC AG Support. Replication between London and Houghton Id 484fd128-d88d-4e67-955f-e6ba14e6d230 IsArchived False IsDefault False IsReadOnly False IsRetentionLockedSla False Name MSSQL-1d-30d-NoArchive-Repl-Houghton-London OwnerOrgName <null> ProtectedObjectCount <null> SnapshotScheduleLastUpdatedAt <null> StateVersion 2 UiColor <null> Version AllOrgsHavingAccess <null> AllOrgsWithAccess <null> ArchivalLocationsUpgradeInfo <null> ArchivalSpec <null> ArchivalSpecs <null> BackupWindows <null> BaseFrequency <null> ClusterToSyncStatusMap <null> FirstFullBackupWindows <null> LocalRetentionLimit <null> LogConfig <null> ObjectSpecificConfigs <null> OwnerOrg <null> PausedClustersInfo <null> ReplicationSpec <null> ReplicationSpecsV2 <null> SnapshotSchedule <null> SourceClusters <null> UpgradeInfo <null> PendingSla <null> ReplicatedObjects <null> CdmId b8db9b86-76e7-409f-a909-b8cbab17b76f CdmLink <null> CdmOnDemandSnapshotCount <null> CopyOnly False DagId 04723631-69a2-55d7-8b5d-b5bef2133910 HasLogConfigFromSla True HasPermissions True HostLogRetention -1 Id 89ed6d72-3d73-5890-a315-6c774ccc5dc5 IsInAvailabilityGroup True IsLogShippingSecondary False IsMount False IsOnline True IsRelic False LogBackupFrequencyInSeconds -2 LogBackupRetentionInHours -3 Name AdventureWorksLT2019 NumWorkloadDescendants 0 OnDemandSnapshotCount <null> PostBackupScript PreBackupScript RecoveryModel <null> ReplicatedObjectCount <null> SlaPauseStatus False UnprotectableReasons <null> Version 16.0.1000.6 AllOrgs <null> AllTags <null> CdmGroupedSnapshots <null> CdmNewestSnapshot <null> CdmOldestSnapshot <null> CdmSnapshots <null> Cluster RubrikSecurityCloud.Types.Cluster CyberEventLockdownMode <null> EosStatus <null> LicensedProducts <null> PauseStatus <null> ProductType <null> RegisteredMode <null> Status <null> StatusFromDb <null> SubStatus <null> SystemStatus <null> Type <null> CdmRbacMigrationStatus <null> ConnectivityLastUpdated <null> DefaultAddress <null> DefaultPort <null> EncryptionEnabled <null> EosDate <null> EstimatedRunway <null> Id 39b92c18-d897-4b55-a7f9-17ff178616d0 IsAirGapped <null> IsClusterRemovalTprEnabled <null> IsHealthy <null> IsTprEnabled <null> LastConnectionTime <null> Name vault-r-london NoSqlWorkloadCount <null> PassesConnectivityCheck <null> RawAddress <null> RegistrationTime <null> SnapshotCount <null> SystemStatusMessage <null> Timezone <null> Version <null> ActivitySeriesConnection <null> AllOrgs <null> AuthorizedOperations <null> CcprovisionInfo <null> CdmClusterNodeDetails <null> CdmNotificationSettings <null> CdmUpgradeInfo <null> CloudInfo <null> ClusterDiskConnection <null> ClusterNodeConnection <null> ClusterNodeStats <null> ConfigProtectionInfo <null> CyberEventLockdownSupportCaseDetails <null> DatagovAutoEnablePolicyConfig <null> DatagovPreviewerConfig <null> GeoLocation <null> GlobalManagerConnectivityStatus <null> IpmiInfo <null> LambdaConfig <null> LambdaFeatureHistory <null> MetadataPullScheduler <null> Metric <null> MetricTimeSeries <null> MetricTimeSeriesNew <null> ReplicationSources <null> ReplicationTargets <null> RubrikSyncStatus <null> SnappableConnection <null> State <null> SystemStatusAffectedNodes <null> WebServerCertificate <null> Vars RubrikSecurityCloud.Types.Cluster+InlineVars ActivitySeriesConnection key value first <null> after <null> last <null> before <null> sortBy <null> filters <null> ClusterDiskConnection key value first <null> after <null> last <null> before <null> filter <null> ClusterNodeConnection key value first <null> after <null> last <null> before <null> filter <null> sortBy <null> sortOrder <null> ClusterNodeStats key valu

# Get SLA Domain Information

In [14]:
$SLADomain = "MSSQL-1d-30d-NoArchive-Repl-Houghton-London"
$RscSlaDomain = Get-RscSla -Name $SLADomain

# Initiate On Demand Snapshot

In [15]:
New-RscMssqlSnapshot -RscMssqlDatabase $RscMssqlDatabase -RscSlaDomain $RscSlaDomain

EndTime,<null>
Id,MSSQL_DB_BACKUP_e88d0ac2-7522-4abd-b3aa-5a286480fb87_3a5fb81a-e41b-49e0-99ea-e356ab8d1768:::0
NodeId,<null>
Progress,<null>
Result,<null>
StartTime,<null>
Status,QUEUED
Error,<null>
Links,<null>


## Track the status of the backup

---

# Disconnect from RSC
It is important to have each script you create end by disconnecting from RSC. If you do not, then you will run into issues with token authentication exhaustion. RSC allows a Service Account to have 10 concurrent open connections to RSC. Additional requests will be denied until existing connections are closed. 

You can easily disconnect your connection by running the below statement. 

In [ ]:
Disconnect-Rsc